# Analisis Data Application Train

Dataset: `application_train.csv` — Home Credit Default Risk  
Tujuan : menyiapkan data hingga siap digunakan untuk training model Machine Learning.

---

## Daftar Isi

| No | Tahap | Cell |
|---|---|---|
| 1 | Data Collection | Library, Load Data, Cek Fitur, Komplemen |
| 2 | Data Preprocessing | Filter Kolom, Cleaning, Isi Missing Value |
| 3 | EDA | Univariat: Pendapatan, Kredit, Cicilan, Outlier, Usia, Registrasi |
| 4 | Visualisasi Data | Jenis Pendapatan, Pekerjaan, Aset, Status Pernikahan |
| 5 | Finalisasi Data | Missing Value, Duplikat, Encoding, Imbalance, Simpan |

---
# [1] DATA COLLECTION

Sumber data yang digunakan adalah dataset siap pakai dari kompetisi Home Credit Default Risk.  
Dataset ini berisi informasi pemohon kredit dari dunia nyata (*real-world observation*).

- `TARGET = 0` : pemohon tidak gagal bayar
- `TARGET = 1` : pemohon gagal bayar

### Library

In [ ]:
# Instalasi library pandas jika belum tersedia di environment
%pip install pandas

In [ ]:
# ============================================================
# IMPORT LIBRARY
# pandas   : membaca dan memanipulasi data tabel
# seaborn  : membuat grafik statistik
# matplotlib : dasar pembuatan grafik
# ============================================================
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


### Melihat isi data CSV

In [ ]:
# ============================================================
# LOAD DATA TRAIN
# Membaca file CSV application_train.csv ke dalam DataFrame.
# DataFrame adalah tabel data yang bisa diolah dengan pandas.
# ============================================================
df_train = pd.read_csv('home-credit-default-risk/application_train.csv')

df_train


In [ ]:
# ============================================================
# HITUNG JUMLAH FITUR (KOLOM)
# Mengetahui berapa banyak informasi yang tersedia di dataset.
# ============================================================
jumlah_fitur_train = len(df_train.columns)

print(f'Jumlah fitur (kolom) pada data train : {jumlah_fitur_train}')


#### Ubah nama kolom ke list

In [ ]:
# ============================================================
# KONVERSI NAMA KOLOM KE DALAM BENTUK LIST
# Berguna untuk melihat semua nama fitur secara lengkap
# dan untuk perbandingan dengan dataset lain.
# ============================================================
nama_fitur_train = df_train.columns.tolist()

nama_fitur_train


### Mencari komplemen antara data train dan test

Komplemen adalah kolom yang ada di train tetapi tidak ada di test.
Kolom inilah yang menjadi **label / target** prediksi kita.
Model hanya bisa dilatih menggunakan data train karena data test tidak memiliki label.

In [ ]:
# ============================================================
# LOAD DATA TEST (SEMENTARA — HANYA UNTUK PERBANDINGAN KOLOM)
# Kita load test sebentar hanya untuk mencari kolom mana
# yang ada di train tapi tidak ada di test.
# ============================================================
df_test_temp = pd.read_csv('home-credit-default-risk/application_test.csv')
nama_fitur_test = df_test_temp.columns.tolist()

# Cari kolom yang ada di train tapi tidak ada di test
komplemen_list = [x for x in nama_fitur_train if x not in nama_fitur_test]

komplemen_list

# TARGET adalah kolom yang hanya ada di train
# Nilai TARGET : 1 = gagal bayar, 0 = tidak gagal bayar


---
# [2] DATA PREPROCESSING

Membersihkan dan menyiapkan data mentah sebelum dianalisis.

Langkah yang dilakukan:
- **Menghapus data rusak** : filter kolom tidak relevan, hapus duplikat
- **Mengisi data kosong** : median untuk numerik, 'Unknown' untuk teks
- **Transformasi data** : konversi tipe data yang tidak sesuai
- **Normalisasi** : tidak diperlukan — model tree-based tidak bergantung pada skala

### Filter kolom

#### train

In [ ]:
# ============================================================
# FILTER KOLOM PADA DATA TRAIN
# Dari 122 kolom yang tersedia, kita pilih hanya kolom-kolom
# yang paling relevan untuk analisis dan pemodelan.
# Kolom yang dipilih mencakup informasi finansial, demografis,
# dan waktu registrasi pemohon.
# TARGET wajib disertakan karena ini adalah label prediksi.
# ============================================================
selected_features = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "NAME_CONTRACT_TYPE",
    "NAME_INCOME_TYPE",
    "CODE_GENDER",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE",
    "ORGANIZATION_TYPE",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH",
    "DAYS_LAST_PHONE_CHANGE",
    "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY",
    "FLAG_MOBIL",
    "FLAG_CONT_MOBILE",
    "TARGET",           # Label: 1 = gagal bayar, 0 = tidak
]

# Filter hanya kolom yang dipilih
df_train_selected = df_train[selected_features]

# Simpan ke file baru di folder InfiniteLearning
df_train_selected.to_csv('home-credit-default-risk/InfiniteLearning/train.csv', index=False)

df_train_selected


### Cleaning data

Setelah filter kolom, kita load kembali data yang sudah disimpan
dan mulai proses pembersihan: cek nilai kosong, isi nilai kosong,
konversi tipe data, dan cek duplikasi.

In [ ]:
# ============================================================
# LOAD DATA TRAIN YANG SUDAH DIFILTER
# Membaca kembali file train yang sudah disimpan.
# ============================================================
df_clean_train = pd.read_csv('home-credit-default-risk/InfiniteLearning/train.csv')

df_clean_train


In [ ]:
# ============================================================
# CEK TOTAL MISSING VALUE
# Missing value adalah sel kosong yang tidak terisi data.
# Nilai ini perlu ditangani agar tidak mengganggu analisis.
# ============================================================
clean_train = df_clean_train.isnull().sum().sum()

print(f'Total data kosong (missing value) pada data train : {clean_train}')


In [ ]:
# ============================================================
# CEK JUMLAH BARIS YANG MEMILIKI MINIMAL 1 MISSING VALUE
# Kita periksa per baris: apakah ada setidaknya satu kolom
# yang kosong pada baris tersebut.
# ============================================================
count_NaN_train_row = df_clean_train.isnull().any(axis=1).sum()

print(f'Jumlah baris pada data train yang memiliki nilai NaN : {count_NaN_train_row}')


#### Penanganan missing value — data train

In [ ]:
# ============================================================
# KOLOM DENGAN MISSING VALUE TERBANYAK
# Mengurutkan kolom berdasarkan jumlah nilai kosong (terbanyak dulu).
# Ini membantu kita memutuskan cara penanganan yang tepat.
# ============================================================
mostMissingValueColumn_train = df_clean_train.isnull().sum().sort_values(ascending=False)

print(f'Fitur yang paling banyak memiliki missing value pada data train :')
print(mostMissingValueColumn_train)


In [ ]:
# ============================================================
# ISI MISSING VALUE PADA KOLOM OCCUPATION_TYPE
# OCCUPATION_TYPE adalah kolom teks (kategorikal).
# Jika kosong, kita isi dengan 'Unknown' agar data tetap lengkap
# tanpa menghapus baris yang mungkin masih berguna.
# ============================================================
df_clean_train['OCCUPATION_TYPE'] = df_clean_train['OCCUPATION_TYPE'].fillna('Unknown')


In [ ]:
# ============================================================
# KONVERSI TIPE DATA AMT_ANNUITY
# Kolom AMT_ANNUITY seharusnya berisi angka (numerik),
# tetapi mungkin terbaca sebagai object (teks) oleh pandas.
# pd.to_numeric akan mengubahnya; nilai yang tidak bisa dikonversi
# akan dijadikan NaN dengan parameter errors='coerce'.
# ============================================================
df_clean_train['AMT_ANNUITY'] = pd.to_numeric(df_clean_train['AMT_ANNUITY'], errors='coerce')


In [ ]:
# ============================================================
# ISI MISSING VALUE PADA AMT_ANNUITY DENGAN MEDIAN
# Setelah konversi, kolom ini mungkin masih memiliki NaN.
# Kita isi menggunakan median (nilai tengah) karena median
# lebih tahan terhadap nilai ekstrem (outlier) dibanding mean.
# ============================================================
df_clean_train['AMT_ANNUITY'] = df_clean_train['AMT_ANNUITY'].fillna(
    df_clean_train['AMT_ANNUITY'].median()
)


In [ ]:
# Verifikasi: pastikan tidak ada lagi missing value setelah proses pembersihan
df_clean_train.isnull().sum().sum()

In [ ]:
# ============================================================
# CEK DATA DUPLIKAT
# Baris duplikat adalah baris yang isinya persis sama dengan
# baris lain. Duplikat bisa memengaruhi hasil analisis.
# ============================================================
df_clean_train.duplicated().sum()


In [ ]:
# ============================================================
# RINGKASAN INFORMASI DATASET
# Menampilkan tipe data tiap kolom, jumlah nilai non-null,
# dan penggunaan memori.
# ============================================================
df_clean_train.info()


In [ ]:
# ============================================================
# BOXPLOT AMT_INCOME_TOTAL — GAMBARAN AWAL OUTLIER
# Boxplot memperlihatkan sebaran nilai: kotak = 50% data tengah,
# garis = median, titik di luar = nilai ekstrem (outlier).
# ============================================================
sns.boxplot(x=df_clean_train['AMT_INCOME_TOTAL'])
plt.show()


---
# [3] EDA — Exploratory Data Analysis

Menganalisis distribusi dan karakteristik fitur numerik secara mendalam.

Analisis yang dilakukan:
- **Analisis Univariat** : distribusi satu variabel (pendapatan, kredit, cicilan, usia)
- **Analisis Outlier** : deteksi nilai ekstrem menggunakan metode IQR
- **Periode Registrasi** : rentang waktu data

### Plotting

#### Distribusi Pendapatan (AMT_INCOME_TOTAL)

Pendapatan total pemohon adalah salah satu fitur terpenting dalam penilaian kredit.
Grafik histogram menunjukkan seberapa sering setiap rentang pendapatan muncul di dataset.

In [ ]:
income = df_clean_train['AMT_INCOME_TOTAL']

# =============================
# STATISTIK PENTING
# =============================
total_pemohon  = len(income)
mean_income    = income.mean()
median_income  = income.median()
std_income     = income.std()
min_income     = income.min()
max_income     = income.max()
q1_income      = income.quantile(0.25)
q3_income      = income.quantile(0.75)
iqr_income     = q3_income - q1_income

# =============================
# VISUALISASI
# =============================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- SUBPLOT 1: Histogram distribusi ---
ax1 = axes[0]
counts, bins, patches = ax1.hist(
    income, bins=50, color='#5dade2', edgecolor='white', linewidth=0.8
)

# Garis rata-rata
ax1.axvline(mean_income, color='#e74c3c', linestyle='--', linewidth=2,
            label=f'Rata-rata: {mean_income:,.0f}')

# Garis median
ax1.axvline(median_income, color='#2ecc71', linestyle='--', linewidth=2,
            label=f'Median: {median_income:,.0f}')

# Garis Q1 dan Q3 (batas kuartil)
ax1.axvline(q1_income, color='#f39c12', linestyle=':', linewidth=1.5,
            label=f'Q1 (25%): {q1_income:,.0f}')
ax1.axvline(q3_income, color='#9b59b6', linestyle=':', linewidth=1.5,
            label=f'Q3 (75%): {q3_income:,.0f}')

ax1.set_title('Distribusi Pendapatan Pemohon Kredit', fontsize=14, fontweight='bold')
ax1.set_xlabel('Jumlah Pendapatan', fontsize=11)
ax1.set_ylabel('Jumlah Pemohon', fontsize=11)
ax1.ticklabel_format(style='plain', axis='x')
ax1.grid(axis='y', linestyle='--', alpha=0.5)
ax1.legend(fontsize=9, loc='upper right')

# --- SUBPLOT 2: Tabel ringkasan statistik ---
ax2 = axes[1]
ax2.axis('off')

table_data = [
    ['Total Pemohon',        f'{total_pemohon:,}'],
    ['Pendapatan Minimum',   f'{min_income:,.0f}'],
    ['Pendapatan Maksimum',  f'{max_income:,.0f}'],
    ['Rata-rata (Mean)',     f'{mean_income:,.0f}'],
    ['Median',               f'{median_income:,.0f}'],
    ['Standar Deviasi',      f'{std_income:,.0f}'],
    ['Kuartil 1 (25%)',      f'{q1_income:,.0f}'],
    ['Kuartil 3 (75%)',      f'{q3_income:,.0f}'],
    ['IQR (Q3 - Q1)',        f'{iqr_income:,.0f}'],
]

table = ax2.table(
    cellText=table_data,
    colLabels=['Statistik', 'Nilai'],
    loc='center',
    cellLoc='left',
    colColours=['#3498db', '#3498db'],
)

# Styling tabel
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)

# Header: teks putih dan tebal
for col_idx in range(2):
    cell = table[0, col_idx]
    cell.set_text_props(color='white', fontweight='bold')

# Warna baris selang-seling untuk kemudahan membaca
for row_idx in range(1, len(table_data) + 1):
    for col_idx in range(2):
        cell = table[row_idx, col_idx]
        if row_idx % 2 == 0:
            cell.set_facecolor('#ebf5fb')
        else:
            cell.set_facecolor('#fdfefe')

ax2.set_title('Ringkasan Statistik Pendapatan', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

# =============================
# CETAK RINGKASAN DI BAWAH
# =============================
print('=' * 50)
print(f"{'RINGKASAN STATISTIK PENDAPATAN':^50}")
print('=' * 50)
print(f'  Total Pemohon          : {total_pemohon:>15,}')
print(f'  Pendapatan Minimum     : {min_income:>15,.0f}')
print(f'  Pendapatan Maksimum    : {max_income:>15,.0f}')
print(f'  Rata-rata (Mean)       : {mean_income:>15,.0f}')
print(f'  Median                 : {median_income:>15,.0f}')
print(f'  Standar Deviasi        : {std_income:>15,.0f}')
print(f'  Kuartil 1 (25%)        : {q1_income:>15,.0f}')
print(f'  Kuartil 3 (75%)        : {q3_income:>15,.0f}')
print(f'  IQR (Q3 - Q1)          : {iqr_income:>15,.0f}')
print('=' * 50)

# =============================
# KESIMPULAN
# =============================
print()
print('KESIMPULAN:')
print(f'  Sebagian besar pemohon memiliki pendapatan antara')
print(f'  Rp {q1_income:,.0f} hingga Rp {q3_income:,.0f} per tahun.')
print(f'  Nilai median (Rp {median_income:,.0f}) lebih rendah dari rata-rata')
print(f'  (Rp {mean_income:,.0f}), artinya ada sebagian kecil pemohon dengan')
print(f'  pendapatan sangat tinggi yang menarik rata-rata ke atas.')
print(f'  Pemohon dengan pendapatan di bawah Rp {q1_income:,.0f} atau')
print(f'  di atas Rp {q3_income:,.0f} tergolong tidak umum di dataset ini.')


#### Distribusi Jumlah Kredit (AMT_CREDIT)

AMT_CREDIT adalah total nilai kredit yang diajukan pemohon.
Melihat distribusinya membantu memahami rentang kredit yang umum di dataset.

In [ ]:
credit = df_clean_train['AMT_CREDIT']

# =============================
# STATISTIK PENTING
# =============================
total_pemohon   = len(credit)
mean_credit     = credit.mean()
median_credit   = credit.median()
std_credit      = credit.std()
min_credit      = credit.min()
max_credit      = credit.max()
q1_credit       = credit.quantile(0.25)
q3_credit       = credit.quantile(0.75)
iqr_credit      = q3_credit - q1_credit

# =============================
# VISUALISASI
# =============================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- SUBPLOT 1: Histogram distribusi ---
ax1 = axes[0]
counts, bins, patches = ax1.hist(
    credit, bins=50, color='#58d68d', edgecolor='white', linewidth=0.8
)

# Garis rata-rata
ax1.axvline(mean_credit, color='#e74c3c', linestyle='--', linewidth=2,
            label=f'Rata-rata: {mean_credit:,.0f}')

# Garis median
ax1.axvline(median_credit, color='#2980b9', linestyle='--', linewidth=2,
            label=f'Median: {median_credit:,.0f}')

# Garis Q1 dan Q3
ax1.axvline(q1_credit, color='#f39c12', linestyle=':', linewidth=1.5,
            label=f'Q1 (25%): {q1_credit:,.0f}')
ax1.axvline(q3_credit, color='#9b59b6', linestyle=':', linewidth=1.5,
            label=f'Q3 (75%): {q3_credit:,.0f}')

ax1.set_title('Distribusi Jumlah Kredit Pemohon', fontsize=14, fontweight='bold')
ax1.set_xlabel('Jumlah Kredit', fontsize=11)
ax1.set_ylabel('Jumlah Pemohon', fontsize=11)
ax1.ticklabel_format(style='plain', axis='x')
ax1.grid(axis='y', linestyle='--', alpha=0.5)
ax1.legend(fontsize=9, loc='upper right')

# --- SUBPLOT 2: Tabel ringkasan statistik ---
ax2 = axes[1]
ax2.axis('off')

table_data = [
    ['Total Pemohon',   f'{total_pemohon:,}'],
    ['Kredit Minimum',  f'{min_credit:,.0f}'],
    ['Kredit Maksimum', f'{max_credit:,.0f}'],
    ['Rata-rata (Mean)',f'{mean_credit:,.0f}'],
    ['Median',          f'{median_credit:,.0f}'],
    ['Standar Deviasi', f'{std_credit:,.0f}'],
    ['Kuartil 1 (25%)', f'{q1_credit:,.0f}'],
    ['Kuartil 3 (75%)', f'{q3_credit:,.0f}'],
    ['IQR (Q3 - Q1)',   f'{iqr_credit:,.0f}'],
]

table = ax2.table(
    cellText=table_data,
    colLabels=['Statistik', 'Nilai'],
    loc='center',
    cellLoc='left',
    colColours=['#27ae60', '#27ae60'],
)

# Styling tabel
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)

# Header: teks putih dan tebal
for col_idx in range(2):
    cell = table[0, col_idx]
    cell.set_text_props(color='white', fontweight='bold')

# Warna baris selang-seling
for row_idx in range(1, len(table_data) + 1):
    for col_idx in range(2):
        cell = table[row_idx, col_idx]
        if row_idx % 2 == 0:
            cell.set_facecolor('#eafaf1')
        else:
            cell.set_facecolor('#fdfefe')

ax2.set_title('Ringkasan Statistik Kredit', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

# =============================
# CETAK RINGKASAN DI BAWAH
# =============================
print('=' * 50)
print(f"{'RINGKASAN STATISTIK KREDIT':^50}")
print('=' * 50)
print(f'  Total Pemohon          : {total_pemohon:>15,}')
print(f'  Kredit Minimum         : {min_credit:>15,.0f}')
print(f'  Kredit Maksimum        : {max_credit:>15,.0f}')
print(f'  Rata-rata (Mean)       : {mean_credit:>15,.0f}')
print(f'  Median                 : {median_credit:>15,.0f}')
print(f'  Standar Deviasi        : {std_credit:>15,.0f}')
print(f'  Kuartil 1 (25%)        : {q1_credit:>15,.0f}')
print(f'  Kuartil 3 (75%)        : {q3_credit:>15,.0f}')
print(f'  IQR (Q3 - Q1)          : {iqr_credit:>15,.0f}')
print('=' * 50)

# =============================
# KESIMPULAN
# =============================
print()
print('KESIMPULAN:')
print(f'  Jumlah kredit yang paling banyak diajukan berkisar antara')
print(f'  Rp {q1_credit:,.0f} hingga Rp {q3_credit:,.0f}.')
print(f'  Median Rp {median_credit:,.0f} berarti setengah pemohon mengajukan')
print(f'  kredit di bawah nilai ini. Rentang yang lebar (Rp {min_credit:,.0f}')
print(f'  hingga Rp {max_credit:,.0f}) menunjukkan variasi kebutuhan')
print(f'  pemohon yang sangat beragam.')


#### Distribusi Cicilan (AMT_ANNUITY)

AMT_ANNUITY adalah besaran cicilan bulanan yang harus dibayar pemohon.
Semakin besar cicilan relatif terhadap pendapatan, semakin tinggi risiko gagal bayar.

In [ ]:
annuity = df_clean_train['AMT_ANNUITY']

# =============================
# STATISTIK PENTING
# =============================
total_pemohon    = len(annuity)
mean_annuity     = annuity.mean()
median_annuity   = annuity.median()
std_annuity      = annuity.std()
min_annuity      = annuity.min()
max_annuity      = annuity.max()
q1_annuity       = annuity.quantile(0.25)
q3_annuity       = annuity.quantile(0.75)
iqr_annuity      = q3_annuity - q1_annuity

# =============================
# VISUALISASI
# =============================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- SUBPLOT 1: Histogram distribusi ---
ax1 = axes[0]
counts, bins, patches = ax1.hist(
    annuity, bins=50, color='#f0b27a', edgecolor='white', linewidth=0.8
)

# Garis rata-rata
ax1.axvline(mean_annuity, color='#e74c3c', linestyle='--', linewidth=2,
            label=f'Rata-rata: {mean_annuity:,.0f}')

# Garis median
ax1.axvline(median_annuity, color='#2980b9', linestyle='--', linewidth=2,
            label=f'Median: {median_annuity:,.0f}')

# Garis Q1 dan Q3
ax1.axvline(q1_annuity, color='#27ae60', linestyle=':', linewidth=1.5,
            label=f'Q1 (25%): {q1_annuity:,.0f}')
ax1.axvline(q3_annuity, color='#9b59b6', linestyle=':', linewidth=1.5,
            label=f'Q3 (75%): {q3_annuity:,.0f}')

ax1.set_title('Distribusi Cicilan Pemohon Kredit', fontsize=14, fontweight='bold')
ax1.set_xlabel('Jumlah Cicilan', fontsize=11)
ax1.set_ylabel('Jumlah Pemohon', fontsize=11)
ax1.ticklabel_format(style='plain', axis='x')
ax1.grid(axis='y', linestyle='--', alpha=0.5)
ax1.legend(fontsize=9, loc='upper right')

# --- SUBPLOT 2: Tabel ringkasan statistik ---
ax2 = axes[1]
ax2.axis('off')

table_data = [
    ['Total Pemohon',    f'{total_pemohon:,}'],
    ['Cicilan Minimum',  f'{min_annuity:,.0f}'],
    ['Cicilan Maksimum', f'{max_annuity:,.0f}'],
    ['Rata-rata (Mean)', f'{mean_annuity:,.0f}'],
    ['Median',           f'{median_annuity:,.0f}'],
    ['Standar Deviasi',  f'{std_annuity:,.0f}'],
    ['Kuartil 1 (25%)',  f'{q1_annuity:,.0f}'],
    ['Kuartil 3 (75%)',  f'{q3_annuity:,.0f}'],
    ['IQR (Q3 - Q1)',    f'{iqr_annuity:,.0f}'],
]

table = ax2.table(
    cellText=table_data,
    colLabels=['Statistik', 'Nilai'],
    loc='center',
    cellLoc='left',
    colColours=['#e67e22', '#e67e22'],
)

# Styling tabel
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)

# Header: teks putih dan tebal
for col_idx in range(2):
    cell = table[0, col_idx]
    cell.set_text_props(color='white', fontweight='bold')

# Warna baris selang-seling
for row_idx in range(1, len(table_data) + 1):
    for col_idx in range(2):
        cell = table[row_idx, col_idx]
        if row_idx % 2 == 0:
            cell.set_facecolor('#fef5e7')
        else:
            cell.set_facecolor('#fdfefe')

ax2.set_title('Ringkasan Statistik Cicilan', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

# =============================
# CETAK RINGKASAN DI BAWAH
# =============================
print('=' * 50)
print(f"{'RINGKASAN STATISTIK CICILAN':^50}")
print('=' * 50)
print(f'  Total Pemohon          : {total_pemohon:>15,}')
print(f'  Cicilan Minimum        : {min_annuity:>15,.0f}')
print(f'  Cicilan Maksimum       : {max_annuity:>15,.0f}')
print(f'  Rata-rata (Mean)       : {mean_annuity:>15,.0f}')
print(f'  Median                 : {median_annuity:>15,.0f}')
print(f'  Standar Deviasi        : {std_annuity:>15,.0f}')
print(f'  Kuartil 1 (25%)        : {q1_annuity:>15,.0f}')
print(f'  Kuartil 3 (75%)        : {q3_annuity:>15,.0f}')
print(f'  IQR (Q3 - Q1)          : {iqr_annuity:>15,.0f}')
print('=' * 50)

# =============================
# KESIMPULAN
# =============================
print()
print('KESIMPULAN:')
print(f'  Cicilan bulanan yang umum berkisar antara Rp {q1_annuity:,.0f}')
print(f'  hingga Rp {q3_annuity:,.0f} per bulan.')
print(f'  Rata-rata cicilan Rp {mean_annuity:,.0f} jika dibandingkan dengan')
print(f'  rata-rata pendapatan, memberikan gambaran beban cicilan')
print(f'  terhadap penghasilan pemohon.')


#### Analisis Outlier Pendapatan

Outlier adalah nilai yang sangat jauh dari nilai-nilai lainnya.
Metode IQR (Interquartile Range) digunakan untuk menentukan batas normal:
- Batas bawah = Q1 - 1.5 x IQR
- Batas atas  = Q3 + 1.5 x IQR

Nilai di luar batas ini dianggap outlier dan perlu diperhatikan.

In [ ]:
income = df_clean_train['AMT_INCOME_TOTAL']

# =============================
# STATISTIK OUTLIER
# =============================
q1 = income.quantile(0.25)
q3 = income.quantile(0.75)
iqr = q3 - q1

# Hitung batas bawah dan batas atas menggunakan metode IQR
batas_bawah = q1 - 1.5 * iqr
batas_atas  = q3 + 1.5 * iqr

# Identifikasi data yang tergolong outlier
outliers       = income[(income < batas_bawah) | (income > batas_atas)]
jumlah_outlier = len(outliers)
persen_outlier = (jumlah_outlier / len(income)) * 100

# =============================
# VISUALISASI
# =============================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- SUBPLOT 1: Boxplot ---
ax1 = axes[0]
bp = sns.boxplot(x=income, color='#85c1e9', ax=ax1, flierprops=dict(
    marker='o', markerfacecolor='#e74c3c', markersize=4, alpha=0.5
))

# Garis batas outlier
ax1.axvline(batas_bawah, color='#e67e22', linestyle=':', linewidth=1.5,
            label=f'Batas Bawah: {batas_bawah:,.0f}')
ax1.axvline(batas_atas, color='#e67e22', linestyle=':', linewidth=1.5,
            label=f'Batas Atas: {batas_atas:,.0f}')

ax1.set_title('Outlier Pendapatan Pemohon Kredit', fontsize=14, fontweight='bold')
ax1.set_xlabel('Jumlah Pendapatan', fontsize=11)
ax1.ticklabel_format(style='plain', axis='x')
ax1.legend(fontsize=9, loc='upper right')
ax1.grid(axis='x', linestyle='--', alpha=0.4)

# --- SUBPLOT 2: Tabel ringkasan outlier ---
ax2 = axes[1]
ax2.axis('off')

table_data = [
    ['Total Pemohon',               f'{len(income):,}'],
    ['Kuartil 1 (Q1)',              f'{q1:,.0f}'],
    ['Kuartil 3 (Q3)',              f'{q3:,.0f}'],
    ['IQR (Q3 - Q1)',               f'{iqr:,.0f}'],
    ['Batas Bawah (Q1 - 1.5*IQR)', f'{batas_bawah:,.0f}'],
    ['Batas Atas (Q3 + 1.5*IQR)',  f'{batas_atas:,.0f}'],
    ['Jumlah Outlier',              f'{jumlah_outlier:,}'],
    ['Persentase Outlier',          f'{persen_outlier:.2f}%'],
    ['Outlier Min',                 f'{outliers.min():,.0f}' if jumlah_outlier > 0 else '-'],
    ['Outlier Max',                 f'{outliers.max():,.0f}' if jumlah_outlier > 0 else '-'],
]

table = ax2.table(
    cellText=table_data,
    colLabels=['Statistik', 'Nilai'],
    loc='center',
    cellLoc='left',
    colColours=['#3498db', '#3498db'],
)

# Styling tabel
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.65)

# Header: teks putih dan tebal
for col_idx in range(2):
    cell = table[0, col_idx]
    cell.set_text_props(color='white', fontweight='bold')

# Warna baris selang-seling; baris outlier diberi highlight merah muda
for row_idx in range(1, len(table_data) + 1):
    for col_idx in range(2):
        cell = table[row_idx, col_idx]
        if row_idx in [7, 8]:                   # Baris jumlah dan persentase outlier
            cell.set_facecolor('#fadbd8')
            cell.set_text_props(fontweight='bold')
        elif row_idx % 2 == 0:
            cell.set_facecolor('#ebf5fb')
        else:
            cell.set_facecolor('#fdfefe')

ax2.set_title('Ringkasan Analisis Outlier', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

# =============================
# CETAK RINGKASAN DI BAWAH
# =============================
print('=' * 55)
print(f"{'ANALISIS OUTLIER PENDAPATAN':^55}")
print('=' * 55)
print(f'  Total Pemohon               : {len(income):>12,}')
print(f'  Kuartil 1 (Q1)              : {q1:>12,.0f}')
print(f'  Kuartil 3 (Q3)              : {q3:>12,.0f}')
print(f'  IQR (Q3 - Q1)               : {iqr:>12,.0f}')
print(f'  Batas Bawah (Q1 - 1.5*IQR) : {batas_bawah:>12,.0f}')
print(f'  Batas Atas  (Q3 + 1.5*IQR) : {batas_atas:>12,.0f}')
print('-' * 55)
print(f'  Jumlah Outlier              : {jumlah_outlier:>12,}')
print(f'  Persentase Outlier          : {persen_outlier:>11.2f}%')
if jumlah_outlier > 0:
    print(f'  Outlier Min                 : {outliers.min():>12,.0f}')
    print(f'  Outlier Max                 : {outliers.max():>12,.0f}')
print('=' * 55)

# =============================
# KESIMPULAN
# =============================
print()
print('KESIMPULAN:')
print(f'  Sebagian besar pemohon memiliki pendapatan antara')
print(f'  Rp {q1_income:,.0f} hingga Rp {q3_income:,.0f} per tahun.')
print(f'  Nilai median (Rp {median_income:,.0f}) lebih rendah dari rata-rata')
print(f'  (Rp {mean_income:,.0f}), artinya ada sebagian kecil pemohon dengan')
print(f'  pendapatan sangat tinggi yang menarik rata-rata ke atas.')
print(f'  Pemohon dengan pendapatan di bawah Rp {q1_income:,.0f} atau')
print(f'  di atas Rp {q3_income:,.0f} tergolong tidak umum di dataset ini.')


---
# [4] VISUALISASI DATA

Menampilkan karakteristik demografis pemohon dalam bentuk grafik dan tabel statistik.

Visualisasi yang ditampilkan:
- Distribusi jenis pendapatan
- 10 pekerjaan terbanyak
- Kepemilikan aset (mobil & rumah)
- Status pernikahan

#### Distribusi Jenis Pendapatan (NAME_INCOME_TYPE)

Jenis pendapatan mencerminkan profil pekerjaan pemohon.
Mengetahui kelompok terbesar membantu memahami karakteristik mayoritas pemohon.

In [ ]:
income_type   = df_clean_train['NAME_INCOME_TYPE'].value_counts()
total_pemohon = len(df_clean_train)

# =============================
# VISUALISASI
# =============================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- SUBPLOT 1: Bar chart ---
ax1 = axes[0]

# Warna: kategori terbesar paling gelap, sisanya gradasi lebih muda
colors = ['#8e44ad' if i == 0 else '#bb8fce' if i == 1 else '#d2b4de' for i in range(len(income_type))]
bars = ax1.bar(range(len(income_type)), income_type.values, color=colors, edgecolor='white', linewidth=0.8)

# Label angka dan persentase di atas setiap batang
for i, (val, pct) in enumerate(zip(income_type.values, (income_type.values / total_pemohon * 100))):
    ax1.text(i, val + total_pemohon * 0.008, f'{val:,}\n({pct:.1f}%)',
             ha='center', va='bottom', fontsize=8, fontweight='bold')

ax1.set_xticks(range(len(income_type)))
ax1.set_xticklabels(income_type.index, rotation=35, ha='right', fontsize=9)
ax1.set_title('Jenis Pendapatan Pemohon Kredit', fontsize=14, fontweight='bold')
ax1.set_xlabel('Jenis Pendapatan', fontsize=11)
ax1.set_ylabel('Jumlah Pemohon', fontsize=11)
ax1.grid(axis='y', linestyle='--', alpha=0.4)
ax1.set_ylim(0, income_type.values[0] * 1.18)

# --- SUBPLOT 2: Tabel ringkasan ---
ax2 = axes[1]
ax2.axis('off')

table_data = []
for idx, (jenis, jumlah) in enumerate(income_type.items()):
    persen = jumlah / total_pemohon * 100
    table_data.append([f'{idx+1}', jenis, f'{jumlah:,}', f'{persen:.2f}%'])

# Baris total di bagian bawah tabel
table_data.append(['', 'TOTAL', f'{total_pemohon:,}', '100.00%'])

table = ax2.table(
    cellText=table_data,
    colLabels=['No', 'Jenis Pendapatan', 'Jumlah', 'Persentase'],
    loc='center',
    cellLoc='left',
    colColours=['#8e44ad', '#8e44ad', '#8e44ad', '#8e44ad'],
)

# Styling tabel
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.7)

# Header: teks putih dan tebal
for col_idx in range(4):
    cell = table[0, col_idx]
    cell.set_text_props(color='white', fontweight='bold')

# Warna baris selang-seling; baris total diberi highlight
total_row = len(table_data)
for row_idx in range(1, total_row + 1):
    for col_idx in range(4):
        cell = table[row_idx, col_idx]
        if row_idx == total_row:
            cell.set_facecolor('#d7bde2')
            cell.set_text_props(fontweight='bold')
        elif row_idx % 2 == 0:
            cell.set_facecolor('#f4ecf7')
        else:
            cell.set_facecolor('#fdfefe')

# Kolom nomor urut rata tengah
for row_idx in range(1, total_row + 1):
    table[row_idx, 0].set_text_props(ha='center')

ax2.set_title('Rincian Jenis Pendapatan', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

# =============================
# CETAK RINGKASAN DI BAWAH
# =============================
print('=' * 60)
print(f"{'RINCIAN JENIS PENDAPATAN PEMOHON':^60}")
print('=' * 60)
print(f"  {'No':<4} {'Jenis Pendapatan':<25} {'Jumlah':>10} {'Persen':>10}")
print('-' * 60)
for idx, (jenis, jumlah) in enumerate(income_type.items()):
    persen = jumlah / total_pemohon * 100
    print(f'  {idx+1:<4} {jenis:<25} {jumlah:>10,} {persen:>9.2f}%')
print('-' * 60)
print(f"  {'':4} {'TOTAL':<25} {total_pemohon:>10,} {'100.00%':>10}")
print('=' * 60)

# =============================
# KESIMPULAN
# =============================
print()
print('KESIMPULAN:')
top_jenis  = income_type.index[0]
top_jumlah = income_type.values[0]
top_persen = top_jumlah / total_pemohon * 100
print(f'  Mayoritas pemohon kredit berpenghasilan dari kategori')
print(f'  "{top_jenis}" sebanyak {top_jumlah:,} orang ({top_persen:.1f}%).')
print(f'  Jenis penghasilan ini mencerminkan profil ekonomi dominan')
print(f'  dari para pemohon di dataset ini.')


#### 10 Pekerjaan Terbanyak (OCCUPATION_TYPE)

Distribusi jenis pekerjaan pemohon memberikan gambaran latar belakang ekonomi.
Kita tampilkan 10 pekerjaan dengan jumlah pemohon terbanyak.

In [ ]:
occupation    = df_clean_train['OCCUPATION_TYPE'].value_counts()
top10         = occupation.head(10)
total_pemohon = len(df_clean_train)
total_kategori = len(occupation)

# =============================
# VISUALISASI
# =============================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- SUBPLOT 1: Horizontal bar chart ---
ax1 = axes[0]

# Warna: urutan 1 paling gelap, selebihnya gradasi
warna = ['#e74c3c' if i == 0 else '#ec7063' if i < 3 else '#f1948a' for i in range(len(top10))]
y_pos = range(len(top10) - 1, -1, -1)  # Urutan terbalik agar terbesar di atas

ax1.barh(list(y_pos), top10.values, color=warna, edgecolor='white', linewidth=0.8)

# Label angka dan persentase di samping setiap batang
for i, val in enumerate(top10.values):
    pct = val / total_pemohon * 100
    ax1.text(val + total_pemohon * 0.003, len(top10) - 1 - i,
             f' {val:,} ({pct:.1f}%)', va='center', fontsize=9, fontweight='bold')

ax1.set_yticks(list(y_pos))
ax1.set_yticklabels(top10.index, fontsize=9)
ax1.set_title('10 Pekerjaan Terbanyak Pemohon Kredit', fontsize=14, fontweight='bold')
ax1.set_xlabel('Jumlah Pemohon', fontsize=11)
ax1.grid(axis='x', linestyle='--', alpha=0.4)
ax1.set_xlim(0, top10.values[0] * 1.30)

# --- SUBPLOT 2: Tabel ringkasan ---
ax2 = axes[1]
ax2.axis('off')

table_data = []
for idx, (jenis, jumlah) in enumerate(top10.items()):
    persen = jumlah / total_pemohon * 100
    table_data.append([f'{idx+1}', jenis, f'{jumlah:,}', f'{persen:.2f}%'])

# Baris 'Lainnya' untuk kategori di luar top 10
sisa        = total_pemohon - top10.sum()
persen_sisa = sisa / total_pemohon * 100
table_data.append(['', f'Lainnya ({total_kategori - 10} kat.)', f'{sisa:,}', f'{persen_sisa:.2f}%'])
table_data.append(['', 'TOTAL', f'{total_pemohon:,}', '100.00%'])

table = ax2.table(
    cellText=table_data,
    colLabels=['No', 'Jenis Pekerjaan', 'Jumlah', 'Persentase'],
    loc='center',
    cellLoc='left',
    colColours=['#c0392b', '#c0392b', '#c0392b', '#c0392b'],
)

# Styling tabel
table.auto_set_font_size(False)
table.set_fontsize(9.5)
table.scale(1, 1.55)

# Header: teks putih dan tebal
for col_idx in range(4):
    table[0, col_idx].set_text_props(color='white', fontweight='bold')

# Warna baris: baris total dan 'lainnya' diberi warna berbeda
total_rows = len(table_data)
for row_idx in range(1, total_rows + 1):
    for col_idx in range(4):
        cell = table[row_idx, col_idx]
        if row_idx == total_rows:
            cell.set_facecolor('#f5b7b1')
            cell.set_text_props(fontweight='bold')
        elif row_idx == total_rows - 1:
            cell.set_facecolor('#fadbd8')
        elif row_idx % 2 == 0:
            cell.set_facecolor('#fdedec')
        else:
            cell.set_facecolor('#fdfefe')

ax2.set_title('Rincian 10 Pekerjaan Teratas', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

# =============================
# CETAK RINGKASAN DI BAWAH
# =============================
print('=' * 65)
print(f"{'10 PEKERJAAN TERBANYAK PEMOHON KREDIT':^65}")
print('=' * 65)
print(f"  {'No':<4} {'Jenis Pekerjaan':<25} {'Jumlah':>10} {'Persen':>10}")
print('-' * 65)
for idx, (jenis, jumlah) in enumerate(top10.items()):
    persen = jumlah / total_pemohon * 100
    print(f'  {idx+1:<4} {jenis:<25} {jumlah:>10,} {persen:>9.2f}%')
print('-' * 65)
print(f"  {'':4} {'Lainnya':<25} {sisa:>10,} {persen_sisa:>9.2f}%")
print(f"  {'':4} {'TOTAL':<25} {total_pemohon:>10,} {'100.00%':>10}")
print('=' * 65)

# =============================
# KESIMPULAN
# =============================
print()
print('KESIMPULAN:')
top_job    = top10.index[0]
top_jml    = top10.values[0]
top_pct    = top_jml / total_pemohon * 100
print(f'  Pekerjaan terbanyak pemohon adalah "{top_job}"')
print(f'  dengan {top_jml:,} orang ({top_pct:.1f}% dari total pemohon).')
print(f'  10 pekerjaan teratas mencakup {top10.sum():,} orang')
persen_top10 = top10.sum() / total_pemohon * 100
print(f'  ({persen_top10:.1f}% dari seluruh pemohon).')
print(f'  Sisanya tersebar di {total_kategori - 10} kategori pekerjaan lainnya.')


#### Distribusi Usia Pemohon (DAYS_BIRTH)

Kolom `DAYS_BIRTH` menyimpan selisih hari antara tanggal lahir dan tanggal pengajuan kredit
dalam bentuk angka negatif. Kita konversi ke usia dalam tahun (positif) agar lebih mudah dibaca.

In [ ]:
# =============================
# HITUNG USIA
# DAYS_BIRTH disimpan sebagai angka negatif (hari mundur dari hari pengajuan).
# Dibagi -365 untuk mengubahnya menjadi usia dalam satuan tahun.
# =============================
usia = df_clean_train['DAYS_BIRTH'] / -365

total_pemohon = len(usia)

# Statistik usia
min_usia    = usia.min()
max_usia    = usia.max()
mean_usia   = usia.mean()
median_usia = usia.median()
std_usia    = usia.std()
q1          = usia.quantile(0.25)
q3          = usia.quantile(0.75)
iqr         = q3 - q1

# =============================
# VISUALISASI
# =============================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- SUBPLOT 1: Histogram dengan kurva KDE ---
# KDE (Kernel Density Estimate) adalah kurva halus yang menunjukkan
# bentuk distribusi secara keseluruhan.
ax1 = axes[0]
sns.histplot(usia, bins=50, kde=True, color='#5dade2', ax=ax1)
ax1.set_title('Distribusi Usia Pemohon Kredit', fontsize=14, fontweight='bold')
ax1.set_xlabel('Usia (Tahun)')
ax1.set_ylabel('Jumlah Pemohon')
ax1.grid(alpha=0.3)

# --- SUBPLOT 2: Tabel statistik ---
ax2 = axes[1]
ax2.axis('off')

table_data = [
    ['Total Pemohon',    f'{total_pemohon:,}'],
    ['Usia Minimum',     f'{min_usia:.1f}'],
    ['Usia Maksimum',    f'{max_usia:.1f}'],
    ['Rata-rata (Mean)', f'{mean_usia:.1f}'],
    ['Median',           f'{median_usia:.1f}'],
    ['Standar Deviasi',  f'{std_usia:.1f}'],
    ['Kuartil 1 (25%)',  f'{q1:.1f}'],
    ['Kuartil 3 (75%)',  f'{q3:.1f}'],
    ['IQR (Q3 - Q1)',    f'{iqr:.1f}'],
]

table = ax2.table(
    cellText=table_data,
    colLabels=['Statistik', 'Nilai'],
    loc='center',
    cellLoc='left',
    colColours=['#2e86c1', '#2e86c1'],
)

# Styling tabel
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.6)

# Header: teks putih dan tebal
for col_idx in range(2):
    table[0, col_idx].set_text_props(color='white', fontweight='bold')

ax2.set_title('Ringkasan Statistik Usia', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

# =============================
# CETAK RINGKASAN DI BAWAH
# =============================
print('=' * 55)
print(f"{'RINGKASAN STATISTIK USIA PEMOHON':^55}")
print('=' * 55)
print(f"{'Total Pemohon':<25}: {total_pemohon:,}")
print(f"{'Usia Minimum':<25}: {min_usia:.1f}")
print(f"{'Usia Maksimum':<25}: {max_usia:.1f}")
print(f"{'Rata-rata (Mean)':<25}: {mean_usia:.1f}")
print(f"{'Median':<25}: {median_usia:.1f}")
print(f"{'Standar Deviasi':<25}: {std_usia:.1f}")
print(f"{'Kuartil 1 (25%)':<25}: {q1:.1f}")
print(f"{'Kuartil 3 (75%)':<25}: {q3:.1f}")
print(f"{'IQR (Q3 - Q1)':<25}: {iqr:.1f}")
print('=' * 55)

# =============================
# KESIMPULAN
# =============================
print()
print('KESIMPULAN:')
print(f'  Pemohon kredit berusia antara {min_usia:.0f} hingga {max_usia:.0f} tahun.')
print(f'  Usia rata-rata {mean_usia:.1f} tahun dengan median {median_usia:.1f} tahun,')
print(f'  artinya setengah pemohon berusia di bawah {median_usia:.1f} tahun.')
print(f'  50% pemohon berada di rentang usia {q1:.1f} hingga {q3:.1f} tahun')
print(f'  — kelompok usia produktif yang menjadi segmen terbesar.')


#### Distribusi Kepemilikan Aset (FLAG_OWN_CAR & FLAG_OWN_REALTY)

Kepemilikan aset seperti mobil dan rumah mencerminkan kondisi finansial pemohon.
Kita gabungkan dua kolom ini untuk melihat kombinasi kepemilikan yang paling umum.

In [ ]:
aset = df_clean_train.copy()

# ============================================================
# MAPPING NILAI Y/N KE LABEL YANG LEBIH DESKRIPTIF
# FLAG_OWN_CAR dan FLAG_OWN_REALTY berisi 'Y' (punya) atau 'N' (tidak).
# Kita ubah ke teks yang lebih mudah dibaca di grafik.
# ============================================================
aset['OWN_CAR']    = aset['FLAG_OWN_CAR'].map({'Y': 'Punya Mobil', 'N': 'Tidak Punya Mobil'})
aset['OWN_REALTY'] = aset['FLAG_OWN_REALTY'].map({'Y': 'Punya Rumah', 'N': 'Tidak Punya Rumah'})

# Gabungkan dua kolom menjadi satu kolom kombinasi
aset['Aset Kombinasi'] = aset['OWN_CAR'] + ' & ' + aset['OWN_REALTY']

kombinasi     = aset['Aset Kombinasi'].value_counts()
total_pemohon = len(df_clean_train)

# =============================
# VISUALISASI
# =============================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- SUBPLOT 1: Bar chart ---
ax1 = axes[0]
warna = ['#2ecc71', '#58d68d', '#82e0aa', '#abebc6']

ax1.bar(kombinasi.index, kombinasi.values, color=warna)

# Label angka dan persentase di atas setiap batang
for i, val in enumerate(kombinasi.values):
    pct = val / total_pemohon * 100
    ax1.text(i, val + total_pemohon * 0.005,
             f'{val:,}\n({pct:.1f}%)',
             ha='center', fontsize=9, fontweight='bold')

ax1.set_title('Distribusi Kepemilikan Aset Pemohon', fontsize=14, fontweight='bold')
ax1.set_xlabel('Kombinasi Kepemilikan Aset')
ax1.set_ylabel('Jumlah Pemohon')
ax1.tick_params(axis='x', rotation=20)
ax1.grid(axis='y', linestyle='--', alpha=0.4)

# --- SUBPLOT 2: Tabel ---
ax2 = axes[1]
ax2.axis('off')

table_data = []
for idx, (jenis, jumlah) in enumerate(kombinasi.items()):
    persen = jumlah / total_pemohon * 100
    table_data.append([idx + 1, jenis, f'{jumlah:,}', f'{persen:.2f}%'])

# Baris total
table_data.append(['', 'TOTAL', f'{total_pemohon:,}', '100.00%'])

table = ax2.table(
    cellText=table_data,
    colLabels=['No', 'Kombinasi Aset', 'Jumlah', 'Persentase'],
    loc='center',
    cellLoc='left',
    colColours=['#27ae60', '#27ae60', '#27ae60', '#27ae60'],
)

# Styling tabel
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.6)

# Header: teks putih dan tebal
for col_idx in range(4):
    table[0, col_idx].set_text_props(color='white', fontweight='bold')

ax2.set_title('Rincian Kepemilikan Aset', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

# =============================
# CETAK RINGKASAN DI BAWAH
# =============================
print('=' * 65)
print(f"{'DISTRIBUSI KEPEMILIKAN ASET PEMOHON':^65}")
print('=' * 65)
print(f"{'No':<4} {'Kombinasi Aset':<35} {'Jumlah':>10} {'Persen':>10}")
print('-' * 65)
for idx, (jenis, jumlah) in enumerate(kombinasi.items()):
    persen = jumlah / total_pemohon * 100
    print(f'{idx+1:<4} {jenis:<35} {jumlah:>10,} {persen:>9.2f}%')
print('-' * 65)
print(f"{'':4} {'TOTAL':<35} {total_pemohon:>10,} {'100.00%':>10}")
print('=' * 65)

# =============================
# KESIMPULAN
# =============================
print()
print('KESIMPULAN:')
print(f'  Pemohon kredit berusia antara {min_usia:.0f} hingga {max_usia:.0f} tahun.')
print(f'  Usia rata-rata {mean_usia:.1f} tahun dengan median {median_usia:.1f} tahun,')
print(f'  artinya setengah pemohon berusia di bawah {median_usia:.1f} tahun.')
print(f'  50% pemohon berada di rentang usia {q1:.1f} hingga {q3:.1f} tahun')
print(f'  — kelompok usia produktif yang menjadi segmen terbesar.')


#### Distribusi Status Pernikahan (NAME_FAMILY_STATUS)

Status pernikahan dapat memengaruhi kondisi finansial pemohon,
misalnya jumlah tanggungan dalam keluarga.

In [ ]:
status        = df_clean_train['NAME_FAMILY_STATUS'].value_counts()
total_pemohon = len(df_clean_train)

# =============================
# VISUALISASI
# =============================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- SUBPLOT 1: Horizontal bar chart ---
ax1 = axes[0]
warna = ['#5dade2', '#76d7c4', '#82e0aa', '#f7dc6f', '#f5b7b1', '#d7bde2']

# Urutan terbalik agar nilai terbesar tampil di atas
ax1.barh(status.index[::-1], status.values[::-1], color=warna)

# Label angka dan persentase di samping setiap batang
for i, val in enumerate(status.values[::-1]):
    pct = val / total_pemohon * 100
    ax1.text(val + total_pemohon * 0.003, i,
             f'{val:,} ({pct:.1f}%)',
             va='center', fontsize=9, fontweight='bold')

ax1.set_title('Distribusi Status Pernikahan Pemohon', fontsize=14, fontweight='bold')
ax1.set_xlabel('Jumlah Pemohon')
ax1.set_ylabel('Status Pernikahan')
ax1.grid(axis='x', linestyle='--', alpha=0.4)

# --- SUBPLOT 2: Tabel ---
ax2 = axes[1]
ax2.axis('off')

table_data = []
for idx, (jenis, jumlah) in enumerate(status.items()):
    persen = jumlah / total_pemohon * 100
    table_data.append([idx + 1, jenis, f'{jumlah:,}', f'{persen:.2f}%'])

# Baris total
table_data.append(['', 'TOTAL', f'{total_pemohon:,}', '100.00%'])

table = ax2.table(
    cellText=table_data,
    colLabels=['No', 'Status Pernikahan', 'Jumlah', 'Persentase'],
    loc='center',
    cellLoc='left',
    colColours=['#2e86c1', '#2e86c1', '#2e86c1', '#2e86c1'],
)

# Styling tabel
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.6)

# Header: teks putih dan tebal
for col_idx in range(4):
    table[0, col_idx].set_text_props(color='white', fontweight='bold')

ax2.set_title('Rincian Status Pernikahan', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

# =============================
# CETAK RINGKASAN DI BAWAH
# =============================
print('=' * 65)
print(f"{'DISTRIBUSI STATUS PERNIKAHAN PEMOHON':^65}")
print('=' * 65)
print(f"{'No':<4} {'Status Pernikahan':<25} {'Jumlah':>10} {'Persen':>10}")
print('-' * 65)
for idx, (jenis, jumlah) in enumerate(status.items()):
    persen = jumlah / total_pemohon * 100
    print(f'{idx+1:<4} {jenis:<25} {jumlah:>10,} {persen:>9.2f}%')
print('-' * 65)
print(f"{'':4} {'TOTAL':<25} {total_pemohon:>10,} {'100.00%':>10}")
print('=' * 65)

# =============================
# KESIMPULAN
# =============================
print()
print('KESIMPULAN:')
print(f'  Pemohon kredit berusia antara {min_usia:.0f} hingga {max_usia:.0f} tahun.')
print(f'  Usia rata-rata {mean_usia:.1f} tahun dengan median {median_usia:.1f} tahun,')
print(f'  artinya setengah pemohon berusia di bawah {median_usia:.1f} tahun.')
print(f'  50% pemohon berada di rentang usia {q1:.1f} hingga {q3:.1f} tahun')
print(f'  — kelompok usia produktif yang menjadi segmen terbesar.')


#### Periode Waktu Registrasi (DAYS_REGISTRATION)

Kolom `DAYS_REGISTRATION` menunjukkan berapa hari lalu pemohon mendaftarkan dokumen identitasnya
sebelum mengajukan kredit. Nilai negatif berarti semakin lama sudah terdaftar.

In [ ]:
# ============================================================
# RANGE WAKTU REGISTRASI PEMOHON
# DAYS_REGISTRATION negatif: semakin besar nilai absolutnya,
# semakin lama dokumen sudah terdaftar sebelum pengajuan kredit.
# ============================================================
min_reg = df_clean_train['DAYS_REGISTRATION'].min()
max_reg = df_clean_train['DAYS_REGISTRATION'].max()

print(f'Periode waktu analisis berdasarkan DAYS_REGISTRATION:')
print(f'Registrasi paling lama : {abs(max_reg):.0f} hari yang lalu')
print(f'Registrasi paling baru : {abs(min_reg):.0f} hari yang lalu')

# =============================
# KESIMPULAN
# =============================
print()
print('KESIMPULAN:')
print(f'  Pemohon kredit berusia antara {min_usia:.0f} hingga {max_usia:.0f} tahun.')
print(f'  Usia rata-rata {mean_usia:.1f} tahun dengan median {median_usia:.1f} tahun,')
print(f'  artinya setengah pemohon berusia di bawah {median_usia:.1f} tahun.')
print(f'  50% pemohon berada di rentang usia {q1:.1f} hingga {q3:.1f} tahun')
print(f'  — kelompok usia produktif yang menjadi segmen terbesar.')


---
# [5] FINALISASI DATA

Mengubah data bersih menjadi data yang benar-benar siap untuk training model Machine Learning.

Langkah yang dilakukan:
- Isi sisa missing value
- Hapus baris duplikat
- Tangani nilai anomali
- Encode kolom teks ke angka *(Feature Engineering: FLAG_TIDAK_BEKERJA)*
- Cek distribusi label
- Pisahkan fitur (X) dan label (y)
- Simpan data final `train_ready.csv`

### Langkah 1 — Isi Sisa Missing Value

Tiga kolom masih memiliki nilai kosong dalam jumlah sangat kecil.
Diisi dengan median agar tidak membuang baris data yang masih berguna.

- `AMT_GOODS_PRICE`       : 278 missing (0.09%)
- `CNT_FAM_MEMBERS`       : 2 missing
- `DAYS_LAST_PHONE_CHANGE`: 1 missing

In [ ]:
# ============================================================
# ISI SISA MISSING VALUE DENGAN MEDIAN
# Median dipilih karena lebih tahan terhadap nilai ekstrem
# dibandingkan mean (rata-rata).
# ============================================================
kolom_sisa_missing = ['AMT_GOODS_PRICE', 'CNT_FAM_MEMBERS', 'DAYS_LAST_PHONE_CHANGE']

for col in kolom_sisa_missing:
    median_val = df_clean_train[col].median()
    df_clean_train[col] = df_clean_train[col].fillna(median_val)
    print(f'  {col:<30} diisi median = {median_val:.2f}')

print()
print(f'Total missing value setelah pengisian: {df_clean_train.isnull().sum().sum()}')


### Langkah 1b — Hapus Baris Duplikat

Baris duplikat adalah baris yang isinya persis sama dengan baris lain.
Duplikat bisa membuat model belajar dari data yang sama berkali-kali
sehingga hasil evaluasi menjadi tidak akurat.

In [ ]:
# ============================================================
# HAPUS BARIS DUPLIKAT
# Baris yang persis sama dihapus — hanya satu yang dipertahankan.
# ============================================================
jumlah_sebelum = len(df_clean_train)

df_clean_train = df_clean_train.drop_duplicates()

jumlah_sesudah  = len(df_clean_train)
jumlah_dihapus  = jumlah_sebelum - jumlah_sesudah

print(f'Baris sebelum  : {jumlah_sebelum:,}')
print(f'Baris dihapus  : {jumlah_dihapus}')
print(f'Baris sesudah  : {jumlah_sesudah:,}')


### Langkah 2 — Tangani Anomali DAYS_EMPLOYED

Nilai `365243` pada kolom `DAYS_EMPLOYED` adalah kode khusus untuk pemohon
yang tidak bekerja atau pensiunan — bukan nilai hari kerja sesungguhnya.
Nilai ini jauh dari rentang normal sehingga bisa menyesatkan model.

Solusi:
- Tambah kolom penanda `FLAG_TIDAK_BEKERJA` (1 = tidak bekerja, 0 = bekerja)
- Ganti nilai anomali dengan 0

In [ ]:
# ============================================================
# TANGANI NILAI ANOMALI PADA DAYS_EMPLOYED
# Nilai 365243 bukan hari kerja nyata, melainkan kode
# untuk pemohon yang tidak bekerja atau pensiunan.
# ============================================================
NILAI_ANOMALI = 365243

jumlah_anomali = (df_clean_train['DAYS_EMPLOYED'] == NILAI_ANOMALI).sum()
print(f'Jumlah baris dengan nilai anomali: {jumlah_anomali:,}')

# Tambah kolom biner sebagai penanda status tidak bekerja
df_clean_train['FLAG_TIDAK_BEKERJA'] = (
    df_clean_train['DAYS_EMPLOYED'] == NILAI_ANOMALI
).astype(int)

# Ganti nilai anomali dengan 0
df_clean_train['DAYS_EMPLOYED'] = df_clean_train['DAYS_EMPLOYED'].replace(NILAI_ANOMALI, 0)

print(f'Nilai anomali diganti dengan 0.')
print(f'Kolom FLAG_TIDAK_BEKERJA ditambahkan.')


### Langkah 3 — Encode Kolom Teks ke Angka

Model Machine Learning hanya bisa memproses angka, bukan teks.
10 kolom bertipe `object` harus dikonversi:
- **Label Encoding**  : untuk kolom dengan 2 nilai unik (binary) — hasilnya 0 dan 1
- **One-Hot Encoding**: untuk kolom dengan banyak nilai — setiap kategori jadi kolom baru bertipe 0/1

In [ ]:
# ============================================================
# IDENTIFIKASI KOLOM TEKS BERDASARKAN JUMLAH NILAI UNIK
# Binary (2 nilai unik)   -> Label Encoding
# Multi-kelas (>2 nilai)  -> One-Hot Encoding
# ============================================================
from sklearn.preprocessing import LabelEncoder

kolom_teks       = df_clean_train.select_dtypes(include='object').columns.tolist()
kolom_binary     = [c for c in kolom_teks if df_clean_train[c].nunique() == 2]
kolom_multiclass = [c for c in kolom_teks if df_clean_train[c].nunique() > 2]

print('Kolom binary (Label Encoding):')
for c in kolom_binary:
    print(f'  {c:<20} -> {df_clean_train[c].unique().tolist()}')

print()
print('Kolom multi-kelas (One-Hot Encoding):')
for c in kolom_multiclass:
    print(f'  {c:<30} ({df_clean_train[c].nunique()} nilai unik)')


In [ ]:
# ============================================================
# LABEL ENCODING — KOLOM BINARY
# Mengubah dua kategori teks menjadi angka 0 dan 1.
# ============================================================
le = LabelEncoder()

for col in kolom_binary:
    df_clean_train[col] = le.fit_transform(df_clean_train[col].astype(str))
    print(f'  {col:<20} -> Label Encoded')

print()

# ============================================================
# ONE-HOT ENCODING — KOLOM MULTI-KELAS
# Setiap kategori menjadi kolom baru dengan nilai 0 atau 1.
# drop_first=True menghapus satu kolom per fitur untuk
# menghindari dummy variable trap (multikolinearitas).
# ============================================================
df_clean_train = pd.get_dummies(df_clean_train, columns=kolom_multiclass, drop_first=True)

print(f'One-Hot Encoding selesai.')
print(f'Jumlah kolom setelah encoding : {df_clean_train.shape[1]}')
print(f'Jumlah baris                  : {df_clean_train.shape[0]:,}')


### Langkah 4 — Cek Distribusi Label (Class Imbalance)

Class imbalance terjadi ketika jumlah satu kelas jauh lebih banyak dari kelas lain.
Jika dibiarkan, model cenderung selalu memprediksi kelas mayoritas.

Strategi yang direkomendasikan saat training:
- `class_weight='balanced'` pada RandomForest atau LogisticRegression
- `scale_pos_weight` pada XGBoost atau LightGBM

In [ ]:
# ============================================================
# CEK DAN VISUALISASI DISTRIBUSI LABEL TARGET
# Rasio > 3:1  : mulai perlu diperhatikan
# Rasio > 10:1 : wajib ditangani sebelum training
# ============================================================
distribusi = df_clean_train['TARGET'].value_counts()
rasio      = df_clean_train['TARGET'].value_counts(normalize=True) * 100

print('Distribusi label TARGET:')
print(f'  Tidak gagal bayar (0) : {distribusi[0]:>8,}  ({rasio[0]:.2f}%)')
print(f'  Gagal bayar       (1) : {distribusi[1]:>8,}  ({rasio[1]:.2f}%)')
print()

rasio_imbalance = distribusi[0] / distribusi[1]
print(f'Rasio ketidakseimbangan: {rasio_imbalance:.1f} : 1')
print()
if rasio_imbalance > 3:
    print('Data tidak seimbang. Gunakan salah satu strategi saat training:')
    print('  - class_weight="balanced" (RandomForest, LogisticRegression)')
    print('  - scale_pos_weight (XGBoost, LightGBM)')

# Visualisasi distribusi label
fig, ax = plt.subplots(figsize=(6, 4))
warna = ['#2ecc71', '#e74c3c']
bars  = ax.bar(
    ['Tidak Gagal Bayar (0)', 'Gagal Bayar (1)'],
    distribusi.values, color=warna, edgecolor='white', width=0.5
)
for bar, val, pct in zip(bars, distribusi.values, rasio.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1000,
        f'{val:,}\n({pct:.1f}%)',
        ha='center', fontweight='bold', fontsize=10
    )
ax.set_title('Distribusi Label TARGET', fontsize=13, fontweight='bold')
ax.set_ylabel('Jumlah Pemohon')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


### Langkah 5 — Pisahkan Fitur (X) dan Label (y)

- **X** : semua kolom informasi yang digunakan model untuk belajar
- **y** : kolom TARGET yang ingin diprediksi (0 = tidak gagal bayar, 1 = gagal bayar)

In [ ]:
# ============================================================
# PISAHKAN FITUR (X) DAN LABEL (y)
# Kolom TARGET dikeluarkan dari X karena ini adalah jawaban,
# bukan informasi input untuk model.
# ============================================================
kolom_drop = [col for col in ['TARGET', 'SK_ID_CURR'] if col in df_clean_train.columns]

y = df_clean_train['TARGET']
X = df_clean_train.drop(columns=kolom_drop)

print(f'Ukuran X (fitur) : {X.shape[0]:,} baris x {X.shape[1]} kolom')
print(f'Ukuran y (label) : {y.shape[0]:,} nilai')
print(f'Missing value    : {X.isnull().sum().sum()}')
print(f'Kolom teks       : {(X.dtypes == "object").sum()} (harus 0 agar siap ML)')
print()
print('Semua kolom sudah numerik dan siap masuk model.')


### Langkah 6 — Simpan Data Final

Data yang sudah sepenuhnya siap disimpan ke file `train_ready.csv`.
File ini yang digunakan langsung sebagai input saat melatih model Machine Learning.

In [ ]:
# ============================================================
# SIMPAN DATA FINAL KE FILE CSV
# Berisi semua fitur yang sudah di-encode + kolom TARGET.
# File ini langsung siap dipakai untuk training model ML.
# ============================================================
import os

os.makedirs('home-credit-default-risk/InfiniteLearning', exist_ok=True)

df_final = X.copy()
df_final['TARGET'] = y.values

output_path = 'home-credit-default-risk/InfiniteLearning/train_ready.csv'
df_final.to_csv(output_path, index=False)

print(f'Tersimpan di     : {output_path}')
print(f'Jumlah baris     : {df_final.shape[0]:,}')
print(f'Jumlah kolom     : {df_final.shape[1]}')
print(f'Missing value    : {df_final.isnull().sum().sum()}')
print(f'Kolom teks       : {(df_final.dtypes == "object").sum()}')
print()
print('Data sudah 100% siap untuk training model Machine Learning.')


In [ ]:
# ============================================================
# RINGKASAN AKHIR SELURUH PROSES
# ============================================================
print('=' * 58)
print(f"{'RINGKASAN DATA TRAIN FINAL':^58}")
print('=' * 58)
print(f'  File sumber    : application_train.csv')
print(f'  File output    : InfiniteLearning/train_ready.csv')
print(f'  Jumlah baris   : {df_final.shape[0]:,}')
print(f'  Jumlah kolom   : {df_final.shape[1]}')
print(f'  Missing value  : {df_final.isnull().sum().sum()}')
print(f'  Duplikat       : {df_final.duplicated().sum()}')
print('-' * 58)
print(f'  Distribusi TARGET:')
for val, count in df_final['TARGET'].value_counts().items():
    pct   = count / len(df_final) * 100
    label = 'Tidak gagal bayar' if val == 0 else 'Gagal bayar      '
    print(f'    {val} ({label}) : {count:,} ({pct:.1f}%)')
print('=' * 58)
print('  Data siap digunakan untuk Machine Learning.')
print('=' * 58)
